In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from mne.decoding import CSP
import mne
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut
from pickle import dump, load 
import sys
import os
import mlflow
import mlflow.sklearn

sys.path.append(os.path.abspath(os.path.join('..')))

from train.train_BCICIV import train_BCICIV
from data_load.load_data_BCICIV import load_all_subjects

mne.set_log_level('WARNING')

## LOAD TRAIN DATA FOR BCICIV DATASET

In [ ]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

## CSP + LDA MODEL

In [ ]:
db_path = "/tmp/mlflow.db"
artifacts_path = "/tmp/mlflow_artifacts"

os.makedirs(artifacts_path, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{db_path}")

experiment_name = "BCI_Motor_Imagery_Classification_Experiment"
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(
        name=experiment_name,
        artifact_location=f"file://{artifacts_path}" 
    )

mlflow.set_experiment(experiment_name)

In [ ]:
transpose = FunctionTransformer(lambda X: np.transpose(X, (0, 2, 1)), validate=False)

pipe1 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

param_grid1 = {
    'csp__n_components': [2, 4, 6],
    'csp__reg': [None, 'ledoit_wolf', 'oas'],
    'clf__solver': ['svd', 'lsqr', 'eigen']
}

In [ ]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="LDA_with_CSP"):

    model, params, best_score = train_BCICIV(data, pipe1, loso, param_grid1)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

## CSP + SVM MODEL

In [ ]:
pipe2 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(log=True, norm_trace=False)),
    ('clf', SVC(kernel='rbf'))
])

param_grid2 = {
    'csp__n_components': [2, 4, 6],
    'csp__reg': [None, 'ledoit_wolf', 'oas'],
    'clf__C': [0.1, 1, 10],
    'clf__gamma': ['scale', 'auto']
}

In [ ]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="SVM_with_CSP"):

    model, params, best_score = train_BCICIV(data, pipe2, loso, param_grid2)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")